## Dependencies

> pip install yfinance, pandas, tensorflow


## Task 1, downloading the financial data
First we download 75 stocks covering a time period of 10 years. The stocks chosen are large and common.

In [6]:
import yfinance as yf
import pandas as pd

# List of 75 stock tickers (example: S&P 500 subset, replace with your own)
tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'JPM', 'V', 'UNH',
    'HD', 'PG', 'MA', 'DIS', 'BAC', 'XOM', 'VZ', 'ADBE', 'CMCSA', 'NFLX',
    'KO', 'PFE', 'CSCO', 'PEP', 'T', 'ABT', 'CVX', 'MRK', 'WMT', 'INTC',
    'CRM', 'MCD', 'NKE', 'WFC', 'MDT', 'COST', 'DHR', 'LLY', 'TMO', 'NEE',
    'TXN', 'LIN', 'UNP', 'HON', 'PM', 'ORCL', 'AMGN', 'IBM', 'QCOM', 'ACN',
    'AVGO', 'LOW', 'SBUX', 'BMY', 'MS', 'RTX', 'INTU', 'GE', 'CAT', 'GS',
    'BLK', 'ISRG', 'PLD', 'MDLZ', 'AMT', 'SPGI', 'SYK', 'CB', 'ZTS', 'DE',
    'MMC', 'ADI', 'NOW', 'GILD', 'TGT', 'LRCX', 'MO', 'ADP', 'CI', 'SCHW'
 ]

# Download 10 years of daily data for all tickers
data = yf.download(tickers, start='2016-03-06', end='2026-03-06', group_by='ticker', auto_adjust=True)
data = data.reset_index() 
data['Date'] = data['Date'].dt.strftime('%Y%m%d')
# Save to CSV (optional)
data.to_csv('stocks_10years.csv')
# Display the first few rows
data.head()

[*********************100%***********************]  80 of 80 completed


Ticker      Date    NFLX                                        AAPL  \
Price               Open    High    Low  Close     Volume       Open   
0       20160307  10.100  10.179  9.525  9.549  238552000  23.189335   
1       20160308   9.500   9.828  9.450  9.623  147970000  22.824699   
2       20160309   9.682   9.837  9.500  9.800  122987000  22.944731   
3       20160310   9.818   9.974  9.545  9.736  170021000  22.967387   
4       20160311   9.951   9.960  9.605  9.766  151143000  23.155361   

Ticker                                   ...         DE                        \
Price        High        Low      Close  ...       Open       High        Low   
0       23.288987  22.865468  23.071566  ...  71.628861  72.771874  71.340994   
1       23.046651  22.738637  22.881319  ...  71.832074  72.077612  70.062523   
2       23.005882  22.709191  22.901701  ...  71.247887  71.713556  69.960934   
3       23.155365  22.682021  22.913031  ...  70.121792  70.189528  68.911044   
4       23.164420  22.987766  23.159891  ...  70.443504  70.638242  69.740767   

Ticker                             MS                                   \
Price       Close   Volume       Open       High        Low      Close   
0       72.543274  3512800  19.581958  19.953287  19.475864  19.771412   
1       70.993866  3212300  19.475866  19.597116  18.899926  18.952972   
2       70.308075  2677200  19.096960  19.142429  18.521020  18.649849   
3       69.749252  2774200  18.877193  19.059068  18.528598  18.680161   
4       70.231834  2779500  19.059063  19.771411  19.028752  19.703207   

Ticker            
Price     Volume  
0        9835300  
1       21715100  
2       23542300  
3       20117300  
4       24457300  

[5 rows x 401 columns]

## Task 2 Dividing it into different sets
Now we divide the data into training, validation and test set. This is achieved by simply dividing the dates and making the data cover different time periods.

In [19]:
test_set = data[data['Date'] >= '20250101']

val_set = data[(data['Date'] >= '20240101') & (data['Date'] <= '20241231')]

train_set = data[data['Date'] < '20240101']

# Show the number of rows in each set
print('Training set:', train_set.shape)
print('Validation set:', val_set.shape)
print('Test set:', test_set.shape)

Training set: (1969, 401)
Validation set: (252, 401)
Test set: (293, 401)


## Task 3 Processing the data
In order to process the data we will add another column according to the instructions. 

In [ ]:
# For each ticker, add a 'Target' column:
# 1 if next day's Close is at least 3% higher than today's Close, 0 otherwise
for ticker in tickers:
    close = data[(ticker, 'Close')]
    next_day_close = close.shift(-1)
    data[(ticker, 'Target')] = (next_day_close >= close * 1.03).astype(int)
    # Set last row's target to 0 (no next day to compare)
    data.loc[data.index[-1], (ticker, 'Target')] = 0

# Recompute the split sets with the new Target column
train_set = data[data[('Date', '')] < '20240101']
val_set = data[(data[('Date', '')] >= '20240101') & (data[('Date', '')] <= '20241231')]
test_set = data[data[('Date', '')] >= '20250101']

Price,Close,Target
0,11.644,1
1,12.068,0
2,12.098,0
3,12.144,1
4,12.670,0
5,12.470,0
6,12.330,0
7,12.474,0
8,12.160,0
9,12.030,0


## Task 4 LSTM
We can now set up and train the LSTM network to minimize the difference between its output and the desired output. First we need to prepare the data so it has the correct format. One important thing to note with this code is that we dont explicitly create a list of errors for each stock and then combine them to a total list of errors. Instead we utilise an array of sample weights to mask out warmup and last-day steps. By settings sample weights to 0 for warmup days and the last day we can ignore these errors. This will achieve the same effect, Keras also automatically averages the loss over the nonzero weights, making the total loss normalised by the number of valid samples.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, TimeDistributed
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.preprocessing import StandardScaler
import numpy as np

WARMUP = 10
FEATURE_COLS = ['Open', 'High', 'Low', 'Close', 'Volume']
NUM_FEATURES = len(FEATURE_COLS)

def prepare_stock_sequences(df, tickers, feature_cols, warmup, scalers=None, fit=False):
    """
    Extract each stock's FULL time series as one sequence.
    Returns:
      X: (N_stocks, T, num_features)
      y: (N_stocks, T, 1) — target at each timestep
      weights: (N_stocks, T) — 0 for warmup & last day, 1 elsewhere
    The LSTM will process the full sequence; warmup outputs are masked from the loss.
    """
    all_X, all_y, valid_tickers = [], [], []
    if scalers is None:
        scalers = {}

    for ticker in tickers:
        try:
            features = df[ticker][feature_cols].values.astype(np.float32)
            targets = df[ticker]['Target'].values.astype(np.float32)
        except KeyError:
            continue

        if np.isnan(features).any() or np.isnan(targets).any():
            continue
        if len(features) <= warmup + 1:
            continue

        if fit:
            scaler = StandardScaler()
            features = scaler.fit_transform(features)
            scalers[ticker] = scaler
        else:
            if ticker not in scalers:
                continue
            features = scalers[ticker].transform(features)

        all_X.append(features)
        all_y.append(targets)
        valid_tickers.append(ticker)

    X = np.array(all_X, dtype=np.float32)               # (N, T, num_features)
    y = np.array(all_y, dtype=np.float32)[..., np.newaxis]  # (N, T, 1)

    # Sample weights: mask warmup period and last timestep (no ground truth)
    T = X.shape[1]
    weights = np.ones((len(valid_tickers), T), dtype=np.float32)
    weights[:, :warmup] = 0.0   # warmup: discard these outputs
    weights[:, -1] = 0.0        # last day: no next-day ground truth

    return X, y, weights, scalers, valid_tickers


# Prepare full-sequence data per stock
X_train, y_train, w_train, scalers, train_tickers = prepare_stock_sequences(
    train_set, tickers, FEATURE_COLS, WARMUP, fit=True
)
X_val, y_val, w_val, _, val_tickers = prepare_stock_sequences(
    val_set, tickers, FEATURE_COLS, WARMUP, scalers=scalers
)
X_test, y_test, w_test, _, test_tickers = prepare_stock_sequences(
    test_set, tickers, FEATURE_COLS, WARMUP, scalers=scalers
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}  (each sample = 1 stock's full series)")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")


def build_and_train(num_units, num_features, X_train, y_train, w_train, X_val, y_val, w_val):
    model = Sequential([
        LSTM(num_units, activation="tanh", return_sequences=True,
             input_shape=(None, num_features)),
        TimeDistributed(Dense(1, activation="sigmoid"))
    ])

    model.compile(optimizer=Adam(), loss="binary_crossentropy")

    checkpoint = ModelCheckpoint(
        f"best_lstm_{num_units}.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    )

    history = model.fit(
        X_train, y_train,
        sample_weight=w_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val, w_val),
        callbacks=[checkpoint, early_stop],
        verbose=1,
    )

    return model, history


[[2.3189335e+01 2.3288986e+01 2.2865467e+01 2.3071566e+01 1.4331560e+08]
 [2.2824699e+01 2.3046650e+01 2.2738638e+01 2.2881319e+01 1.2624760e+08]
 [2.2944731e+01 2.3005882e+01 2.2709190e+01 2.2901701e+01 1.0880680e+08]
 ...
 [1.9051089e+02 1.9151050e+02 1.8912527e+02 1.9116409e+02 4.8087700e+07]
 [1.9214392e+02 1.9265857e+02 1.9118388e+02 1.9158968e+02 3.4049900e+07]
 [1.9190640e+02 1.9240126e+02 1.8975871e+02 1.9055049e+02 4.2672100e+07]]
[[4.5367085e+01 4.5578259e+01 4.4504795e+01 4.4900742e+01 3.8407800e+07]
 [4.4698353e+01 4.5868607e+01 4.4522373e+01 4.5446259e+01 3.3835100e+07]
 [4.5657452e+01 4.6502144e+01 4.5631058e+01 4.6493347e+01 2.8251600e+07]
 ...
 [3.6736172e+02 3.6870853e+02 3.6649661e+02 3.6773529e+02 1.4905400e+07]
 [3.6901324e+02 3.7008478e+02 3.6782376e+02 3.6892477e+02 1.4327000e+07]
 [3.6963254e+02 3.7077289e+02 3.6715524e+02 3.6967188e+02 1.8730800e+07]]
[[3.5985603e+01 3.6087833e+01 3.4992607e+01 3.5372734e+01 6.1690000e+07]
 [3.5153889e+01 3.5841694e+01 3.4936039

### Threshold tuning and test set evaluation
We use the validation set to find the best amount of units for the LSTM output. We also implement an early stopping function if we dont see any improvement for a set amount of epohcs. Then we evaluate on the test set by computing the average profit per day across all stocks and time steps.

<br>

Best model: 64 units (val_loss=0.1343)
=== Test Set Results ===
Threshold: 0.15
Predictions made: 579 / 22560
Average profit per day (normalized by N×M): 0.000045

In [45]:
unit_options = [8, 16, 32, 64]

best_val_loss = float("inf")
best_units = None
best_model = None

# Train models with different unit sizes, select by lowest validation loss
for num_units in unit_options:
    print(f"\n--- Training LSTM with {num_units} units ---")
    model, history = build_and_train(
        num_units, NUM_FEATURES, X_train, y_train, w_train, X_val, y_val, w_val
    )
    val_loss = min(history.history["val_loss"])
    print(f"LSTM ({num_units} units) - Best Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_units = num_units
        best_model = model

print(f"\nBest model: {best_units} units (val_loss={best_val_loss:.4f})")

# --- Threshold tuning on validation set ---
val_preds = best_model.predict(X_val)  # (N_val, T_val, 1)

# Get raw (un-normalized) Close prices for profit calculation
val_close = np.array([
    val_set[ticker]['Close'].values.astype(np.float64) for ticker in val_tickers
])  # (N_val, T_val)

N_val = len(val_tickers)
T_val = X_val.shape[1]
M_val = T_val - WARMUP - 1  # timesteps considered per stock (after warmup, excluding last)

thresholds = np.arange(0.05, 1.0, 0.05)
best_threshold = 0.5
best_avg_profit = -float("inf")

for thresh in thresholds:
    total_profit = 0.0
    for i in range(N_val):
        for t in range(WARMUP, T_val - 1):  # after warmup, up to second-to-last day
            if val_preds[i, t, 0] >= thresh:
                profit = (val_close[i, t + 1] / val_close[i, t]) - 1
                total_profit += profit
    avg_profit = total_profit / (N_val * M_val)  # normalize by N × M
    if avg_profit > best_avg_profit:
        best_avg_profit = avg_profit
        best_threshold = thresh

print(f"\nBest threshold: {best_threshold:.2f} (avg profit on val: {best_avg_profit:.6f})")

# --- Test set evaluation (no tuning, final evaluation only) ---
test_preds = best_model.predict(X_test)  # (N_test, T_test, 1)

test_close = np.array([
    test_set[ticker]['Close'].values.astype(np.float64) for ticker in test_tickers
])  # (N_test, T_test)

N_test = len(test_tickers)
T_test = X_test.shape[1]
M_test = T_test - WARMUP - 1

total_profit = 0.0
num_predictions = 0

for i in range(N_test):
    for t in range(WARMUP, T_test - 1):
        if test_preds[i, t, 0] >= best_threshold:
            profit = (test_close[i, t + 1] / test_close[i, t]) - 1
            total_profit += profit
            num_predictions += 1

avg_profit = total_profit / (N_test * M_test)

print(f"\n=== Test Set Results ===")
print(f"Threshold: {best_threshold:.2f}")
print(f"Predictions made: {num_predictions} / {N_test * M_test}")
print(f"Average profit per day (normalized by N×M): {avg_profit:.6f}")



--- Training LSTM with 8 units ---
Epoch 1/100


c:\Users\jarld\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - loss: 0.6972 - val_loss: 0.6028
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 614ms/step - loss: 0.6827 - val_loss: 0.5969
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 616ms/step - loss: 0.6687 - val_loss: 0.5905
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 610ms/step - loss: 0.6549 - val_loss: 0.5838
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 573ms/step - loss: 0.6411 - val_loss: 0.5762
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 576ms/step - loss: 0.6272 - val_loss: 0.5674
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 655ms/step - loss: 0.6131 - val_loss: 0.5570
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 614ms/step - loss: 0.5988 - val_loss: 0.5451
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 583ms/step - loss: 0.5841 - val_loss: 0.5318
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 654ms/step - loss: 0.5694 - val_loss: 0.5175
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 638ms/step - loss: 0.5544 - val_loss: 0.5026
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 618ms/step - loss: 0.5394 - val_loss

## Task 5 Rule based system
We implement an evolvable rule-based system where each rule is of the form IF(condition), with conditions involving ratios of prices or volumes at different days. Rules are organized into groups (conjunction/AND): a group outputs 1 only if ALL its rules are True. The system is a disjunction (OR) of groups: it outputs 1 if ANY group outputs 1. An evolutionary algorithm evolves both the rule parameters and the structure (number of rules per group, number of groups). We run the EA with several different parameter settings, select the best system using holdout validation, and then evaluate on the test set.

In [ ]:
import random
import copy

MAX_OFFSET = 10  # Maximum lookback in days for rule conditions

# --- Extract raw Close and Volume data per stock ---
def extract_raw_data(df, ticker_list):
    close_data, volume_data, valid = [], [], []
    for ticker in ticker_list:
        try:
            c = df[ticker]['Close'].values.astype(np.float64)
            v = df[ticker]['Volume'].values.astype(np.float64)
        except KeyError:
            continue
        if np.isnan(c).any() or np.isnan(v).any():
            continue
        if len(c) <= WARMUP + 1:
            continue
        close_data.append(c)
        volume_data.append(v)
        valid.append(ticker)
    return close_data, volume_data, valid

close_train, vol_train, rbs_train_tickers = extract_raw_data(train_set, tickers)
close_val, vol_val, rbs_val_tickers = extract_raw_data(val_set, tickers)
close_test, vol_test, rbs_test_tickers = extract_raw_data(test_set, tickers)

print(f"Rule-based system data: {len(rbs_train_tickers)} train, "
      f"{len(rbs_val_tickers)} val, {len(rbs_test_tickers)} test stocks")


# --- Rule: IF feature[t-a] / feature[t-b] > (or <) threshold ---
class Rule:
    def __init__(self, feature_type, offset_a, offset_b, threshold, direction):
        self.feature_type = feature_type
        self.offset_a = offset_a
        self.offset_b = offset_b
        self.threshold = threshold
        self.direction = direction

    def copy(self):
        return Rule(self.feature_type, self.offset_a, self.offset_b,
                    self.threshold, self.direction)


class RuleGroup:
    """Conjunction (AND) of rules. Outputs True only if ALL rules are True."""
    def __init__(self, rules):
        self.rules = list(rules)

    def copy(self):
        return RuleGroup([r.copy() for r in self.rules])


class RuleSystem:
    """Disjunction (OR) of groups. Outputs 1 if ANY group is True."""
    def __init__(self, groups):
        self.groups = list(groups)

    def copy(self):
        return RuleSystem([g.copy() for g in self.groups])


# --- Random initialization ---
def random_rule():
    return Rule(
        feature_type=random.choice(['price', 'volume']),
        offset_a=random.randint(0, MAX_OFFSET),
        offset_b=random.randint(0, MAX_OFFSET),
        threshold=random.uniform(0.85, 1.15),
        direction=random.choice(['>', '<'])
    )

def random_group(max_rules=4):
    return RuleGroup([random_rule() for _ in range(random.randint(1, max_rules))])

def random_system(max_groups=4, max_rules=4):
    return RuleSystem([random_group(max_rules) for _ in range(random.randint(1, max_groups))])


# --- Vectorized evaluation (numpy) for speed ---
def eval_system_on_stock(system, close, volume, warmup):
    """Evaluate system on one stock. Returns (output_mask, profits) arrays."""
    T = len(close)
    t_range = np.arange(warmup, T - 1)
    n = len(t_range)
    if n == 0:
        return np.array([], dtype=bool), np.array([])

    sys_out = np.zeros(n, dtype=bool)

    for group in system.groups:
        grp_out = np.ones(n, dtype=bool)
        for rule in group.rules:
            vals = close if rule.feature_type == 'price' else volume
            a_idx = t_range - rule.offset_a
            b_idx = t_range - rule.offset_b
            valid = (a_idx >= 0) & (b_idx >= 0)
            rule_out = np.zeros(n, dtype=bool)
            b_vals = vals[np.clip(b_idx, 0, T - 1)]
            valid &= (b_vals != 0)
            if valid.any():
                ratios = vals[a_idx[valid]] / vals[b_idx[valid]]
                if rule.direction == '>':
                    rule_out[valid] = ratios > rule.threshold
                else:
                    rule_out[valid] = ratios < rule.threshold
            grp_out &= rule_out
        sys_out |= grp_out

    profits = close[t_range + 1] / close[t_range] - 1
    return sys_out, profits


def compute_fitness(system, close_data, vol_data, warmup):
    """Average profit per stock-day, normalized by N × M."""
    total_profit = 0.0
    total_M = 0
    for close, vol in zip(close_data, vol_data):
        out, profits = eval_system_on_stock(system, close, vol, warmup)
        if len(out) > 0:
            total_profit += profits[out].sum()
            total_M += len(out)
    return total_profit / total_M if total_M > 0 else 0.0


# --- Mutation operators ---
def mutate(system, max_rules=4):
    s = system.copy()
    for group in s.groups:
        # Mutate a random rule's parameters
        if group.rules and random.random() < 0.5:
            r = group.rules[random.randint(0, len(group.rules) - 1)]
            what = random.choice(['feature', 'offset_a', 'offset_b', 'threshold', 'direction'])
            if what == 'feature':
                r.feature_type = random.choice(['price', 'volume'])
            elif what == 'offset_a':
                r.offset_a = random.randint(0, MAX_OFFSET)
            elif what == 'offset_b':
                r.offset_b = random.randint(0, MAX_OFFSET)
            elif what == 'threshold':
                r.threshold += random.gauss(0, 0.05)
            else:
                r.direction = '<' if r.direction == '>' else '>'
        # Add a new rule to the group
        if random.random() < 0.15:
            group.rules.append(random_rule())
        # Remove a rule from the group
        if len(group.rules) > 1 and random.random() < 0.15:
            group.rules.pop(random.randint(0, len(group.rules) - 1))
    # Add a new group
    if random.random() < 0.1:
        s.groups.append(random_group(max_rules))
    # Remove a group
    if len(s.groups) > 1 and random.random() < 0.1:
        s.groups.pop(random.randint(0, len(s.groups) - 1))
    return s


# --- Evolutionary Algorithm ---
def run_ea(close_train, vol_train, close_val, vol_val, warmup,
           pop_size=50, generations=100, max_groups=4, max_rules=4, tournament_k=3):
    population = [random_system(max_groups, max_rules) for _ in range(pop_size)]
    best_system = None
    best_val_profit = -float('inf')

    for gen in range(generations):
        # Evaluate fitness on training data
        fitnesses = [compute_fitness(s, close_train, vol_train, warmup) for s in population]

        # Evaluate top candidates on validation set
        top_indices = sorted(range(pop_size), key=lambda i: fitnesses[i], reverse=True)[:5]
        for idx in top_indices:
            vp = compute_fitness(population[idx], close_val, vol_val, warmup)
            if vp > best_val_profit:
                best_val_profit = vp
                best_system = population[idx].copy()

        if gen % 25 == 0:
            best_train = max(fitnesses)
            print(f"  Gen {gen}/{generations}: best train = {best_train:.6f}, best val = {best_val_profit:.6f}")

        # Selection + mutation for next generation
        new_pop = []
        elite_idx = max(range(pop_size), key=lambda i: fitnesses[i])
        new_pop.append(population[elite_idx].copy())  # elitism

        while len(new_pop) < pop_size:
            tourn = random.sample(range(pop_size), min(tournament_k, pop_size))
            winner = max(tourn, key=lambda i: fitnesses[i])
            new_pop.append(mutate(population[winner], max_rules))

        population = new_pop

    return best_system, best_val_profit


# --- Run EA with several different parameter settings ---
configs = [
    {"pop_size": 50,  "generations": 100, "max_groups": 3, "max_rules": 3, "tournament_k": 3},
    {"pop_size": 80,  "generations": 100, "max_groups": 5, "max_rules": 4, "tournament_k": 5},
    {"pop_size": 50,  "generations": 150, "max_groups": 4, "max_rules": 5, "tournament_k": 3},
]

overall_best_system = None
overall_best_profit = -float('inf')

for i, cfg in enumerate(configs):
    print(f"\n=== EA Run {i+1}/{len(configs)} — pop={cfg['pop_size']}, "
          f"gens={cfg['generations']}, groups≤{cfg['max_groups']}, rules≤{cfg['max_rules']} ===")
    system, val_profit = run_ea(close_train, vol_train, close_val, vol_val, WARMUP, **cfg)
    print(f"  Final best val profit: {val_profit:.6f}")
    if val_profit > overall_best_profit:
        overall_best_profit = val_profit
        overall_best_system = system

print(f"\nBest overall validation profit: {overall_best_profit:.6f}")

# --- Evaluate best system on test set ---
total_profit = 0.0
num_predictions = 0
total_M = 0

for close, vol in zip(close_test, vol_test):
    out, profits = eval_system_on_stock(overall_best_system, close, vol, WARMUP)
    if len(out) > 0:
        total_profit += profits[out].sum()
        num_predictions += int(out.sum())
        total_M += len(out)

avg_profit = total_profit / total_M if total_M > 0 else 0.0

print(f"\n=== Test Set Results (Rule-Based System) ===")
print(f"Predictions made: {num_predictions} / {total_M}")
print(f"Average profit per day (normalized by N×M): {avg_profit:.6f}")

print(f"\nBest system structure: {len(overall_best_system.groups)} group(s)")
for gi, group in enumerate(overall_best_system.groups):
    print(f"  Group {gi+1} ({len(group.rules)} rule(s)):")
    for ri, rule in enumerate(group.rules):
        print(f"    Rule {ri+1}: {rule.feature_type}[t-{rule.offset_a}] / "
              f"{rule.feature_type}[t-{rule.offset_b}] {rule.direction} {rule.threshold:.4f}")

Rule-based system data: 80 train, 80 val, 80 test stocks

=== EA Run 1/3 — pop=50, gens=100, groups≤3, rules≤3 ===
  Gen 0/100: best train = 0.000764, best val = 0.000830
  Gen 25/100: best train = 0.000877, best val = 0.000830
  Gen 50/100: best train = 0.000897, best val = 0.000830
  Gen 75/100: best train = 0.000935, best val = 0.000830
  Final best val profit: 0.000841

=== EA Run 2/3 — pop=80, gens=100, groups≤5, rules≤4 ===
  Gen 0/100: best train = 0.000806, best val = 0.000821
  Gen 25/100: best train = 0.000919, best val = 0.000832
  Gen 50/100: best train = 0.000936, best val = 0.000862
  Gen 75/100: best train = 0.000968, best val = 0.000862
  Final best val profit: 0.000862

=== EA Run 3/3 — pop=50, gens=150, groups≤4, rules≤5 ===
  Gen 0/150: best train = 0.000755, best val = 0.000821
  Gen 25/150: best train = 0.000920, best val = 0.000821
  Gen 50/150: best train = 0.000929, best val = 0.000824
  Gen 75/150: best train = 0.000941, best val = 0.000825
  Gen 100/150: best 

## Results

Best overall validation profit: 0.000862

=== Test Set Results (Rule-Based System) ===
Predictions made: 21069 / 22560
Average profit per day (normalized by N×M): 0.000607

## Task 6 Overall discussion
The results are quite surprising since it seems that the rule based system outperforms the LSTM. Thus, in this specific case there seems to be no benefit to choose the LSTM since the rule based system offers transperency and better performance. This is likely because the dataset used is relatively small and LSTMs generally require a lot of data to generalize well and are also prone to overfitting when data is limited. Another issue is that there are few stocks that increase with the threshold of 3% per day which we defined in the beginning, thus limiting the amount of data to the LSTM even more. 

Overall, according to research, LSTM systems are supposed to offer better performence. In which case the choice between the two systems is harder and one must consider both of their benefits and disadvantages. A key benefit of rule-based systems is their transparency and interpretability. Each rule is explicit and human-readable, so we can easily see the logic behind every decision the system makes. This allows us to understand, audit, and even modify the system’s behavior based on domain knowledge or new insights.